In [1]:
import sys
!{sys.executable} -m pip install google-cloud-bigquery db-dtypes pandas-gbq gcsfs

In [3]:
import sys
!{sys.executable} -m pip install google-cloud-bigquery db-dtypes pandas-gbq gcsfs

In [3]:
from google.cloud import bigquery
import pandas as pd

PROJECT_ID = "mdff-practicum-2026"
DATASET = "mdff"

client = bigquery.Client(project=PROJECT_ID)
print("Connected:", client.project)

Connected: mdff-practicum-2026


In [15]:
tables = list(client.list_tables("mdff-practicum-2026.mdff"))
inventory = [(t.table_id, t.full_table_id) for t in tables]
df_inventory = pd.DataFrame(inventory, columns=["table_name", "full_table_id"])
df_inventory = df_inventory.sort_values("table_name").reset_index(drop=True)
df_inventory

,table_name,full_table_id
0,DECEASED_DONOR_DATA,mdff-practicum-2026:mdff.DECEASED_DONOR_DATA
1,DECEASED_DONOR_DCD_MEASURES,mdff-practicum-2026:mdff.DECEASED_DONOR_DCD_ME...
2,DECEASED_DONOR_FORMATS_FLATFILE,mdff-practicum-2026:mdff.DECEASED_DONOR_FORMAT...
3,DECEASED_DONOR_INOTROPIC_MEDS,mdff-practicum-2026:mdff.DECEASED_DONOR_INOTRO...
4,IE_DATA,mdff-practicum-2026:mdff.IE_DATA
...,...,...
66,txf_ki,mdff-practicum-2026:mdff.txf_ki
67,txf_kp,mdff-practicum-2026:mdff.txf_kp
68,txf_li,mdff-practicum-2026:mdff.txf_li
69,txf_lu,mdff-practicum-2026:mdff.txf_lu


In [17]:
rows = []
for t in tables:
    full = client.get_table(t.reference)
    rows.append((full.table_id, full.num_rows, round(full.num_bytes / 1e6, 1)))

df_inventory = pd.DataFrame(rows, columns=["table_name", "row_count", "size_mb"])
df_inventory = df_inventory.sort_values("row_count", ascending=False).reset_index(drop=True)
df_inventory

,table_name,row_count,size_mb
0,KIDPAN_WLHISTORY_DATA,47016205,12048.5
1,fol_immuno,9262786,435.1
2,KIDPAN_IMMUNO_FOLLOWUP_DATA,5514999,2534.9
3,KIDNEY_FOLLOWUP_DATA,5134196,1019.0
4,txf_ki,5076988,909.6
...,...,...,...
66,institution,724,0.1
67,LIVING_DONOR_FORMATS_FLATFILE,517,0.0
68,hist_opo_txc,446,0.0
69,NON_RECOV_DCD_MEASURES,425,0.0


In [5]:
tables = list(client.list_tables(f"{PROJECT_ID}.{DATASET}"))
rows = []
for t in tables:
    full = client.get_table(t.reference)
    rows.append((full.table_id, full.num_rows, round(full.num_bytes / 1e6, 1)))

df_inventory = pd.DataFrame(rows, columns=["table_name", "row_count", "size_mb"])
df_inventory = df_inventory.sort_values("row_count", ascending=False).reset_index(drop=True)
df_inventory

,table_name,row_count,size_mb
0,KIDPAN_WLHISTORY_DATA,47016205,12048.5
1,fol_immuno,9262786,435.1
2,KIDPAN_IMMUNO_FOLLOWUP_DATA,5514999,2534.9
3,KIDNEY_FOLLOWUP_DATA,5134196,1019.0
4,txf_ki,5076988,909.6
...,...,...,...
66,institution,724,0.1
67,LIVING_DONOR_FORMATS_FLATFILE,517,0.0
68,hist_opo_txc,446,0.0
69,NON_RECOV_DCD_MEASURES,425,0.0


In [7]:
query = """
SELECT *
FROM `mdff-practicum-2026.mdff.cand_kipa`
LIMIT 5
"""
df = client.query(query).to_dataframe()
df.head()

/opt/anaconda3/lib/python3.12/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,PERS_ID,PX_ID,WL_ORG,CAN_GENDER,CAN_ABO,CAN_RACE,CAN_RACE_SRTR,CAN_ETHNICITY_SRTR,DON_TY,REC_TX_PROCEDURE_TY,...,CAN_INIT_ACT_STAT_DT,CAN_INIT_INACT_STAT_DT,CAN_LAST_ACT_STAT_DT,CAN_LAST_INACT_STAT_DT,CAN_INIT_ACT_STAT_CD,PERS_NEXTTX,PERS_NEXTTX_TRR_ID,CAN_TRR_EXISTS,CAN_PAIRED_LIVE_DONOR_ID,DONOR_ID
0,5191145.0,1318333.0,KI,F,A,None,WHITE,LATINO,None,None,...,None,2020-01-10,None,2021-03-27,None,None,None,0.0,None,None
1,8393025.0,1306510.0,KI,M,O,None,WHITE,LATINO,None,None,...,2019-11-01,None,2023-02-09,None,4010.0,None,None,0.0,None,None
2,8417206.0,1328303.0,KI,M,O,8.0,WHITE,NLATIN,None,None,...,None,2020-03-04,None,2021-06-17,None,None,None,0.0,None,None
3,8402511.0,1311878.0,KI,F,O,8.0,WHITE,NLATIN,None,None,...,2019-12-02,2021-09-02,2021-09-01,2024-11-13,4010.0,None,None,0.0,None,None
4,8416443.0,1332839.0,KI,M,O,8.0,WHITE,NLATIN,None,None,...,2020-03-27,None,2023-03-03,None,4010.0,2023-02-26,988262.0,0.0,None,None
